# DACON 공식 Baseline + DF-Arena 1B + HTDemucs + AASIST 전체 Ensemble

이 노트북은 마지막 `RawBoost_SONICS_DynamicMix_25000_DACON.ipynb`가 만든
`exp03_aasist_rawboost/best.pt`를 DACON 공식 베이스라인에 **추가**합니다.

```text
원본 오디오
├─ PANNs Cnn14 ───────────────────────────────┐
│                                             ├─ Presence logit ensemble
├─ AASIST 5-head best.pt ─────────────────────┤
│                                             ├─ File/Voice/Music ensemble
└─ HTDemucs(시간 예산 적용)                    │
   ├─ vocals ────────> DF-Arena 1B ───────────┤
   └─ accompaniment -> DF-Arena 1B ───────────┘
```

노트북의 역할은 세 가지입니다.

1. 학습과 출처가 겹치지 않는 네 source pool을 각각 625개씩 선택합니다.
2. DACON 평가 데이터 특성에 맞춰 길이 4~60초, 16 kHz, mono/stereo, 여러 codec과 전화채널을 포함하는 고정 OOD DynamicMix 2,500개를 만들고 공식 지표로 평가합니다.
3. 공식 baseline 자산을 유지하면서 AASIST를 추가한 `submit.zip`을 생성합니다.

중요한 제한:

- OOD 2,500개는 최종 평가용입니다. 결과를 보고 ensemble weight를 반복 조정하면 독립 test가 아닙니다.
- 50분은 코드만으로 보장할 수 없습니다. 이 노트북은 L4 실측값에 15% 안전계수를 적용하고 50분을 넘으면 ZIP 생성을 중단합니다.
- SpeechFake 전체는 매우 커서 선택 다운로드가 어렵습니다. 요청한 엄격 OOD 기본값은 사용자가 선별한 SpeechFake test 두 폴더이며, 자동 실행이 필요할 때만 8.16 GB 단일 ZIP인 In-the-Wild로 바꿀 수 있습니다.

공식 자료: [평가/제출 규격](https://dacon.io/competitions/official/236749/overview/evaluation), [공식 baseline](https://dacon.io/competitions/official/236749/codeshare/14153), [DF-Arena 1B](https://huggingface.co/Speech-Arena-2025/DF_Arena_1B_V_1)


## 0. Colab L4 런타임과 Google Drive

Colab에서 런타임 유형을 **L4 GPU**로 선택한 뒤 실행하세요. DACON 평가 서버 기본 패키지를 최대한 그대로 사용합니다.


In [ ]:
!nvidia-smi
!pip -q install "huggingface_hub==0.34.4" "panns-inference==0.1.1" "demucs==4.0.1" "librosa==0.10.2.post1" "soundfile==0.12.1"


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


## 1. 설정

`BASELINE_OR_OPEN_ZIP`에는 DACON 데이터 탭에서 받은 `open.zip` 또는 코드 공유에서 받은 원본 `baseline_submit.zip`을 지정합니다. 로그인과 약관 동의가 필요하므로 노트북이 DACON 파일을 대신 다운로드하지 않습니다.


In [ ]:
from __future__ import annotations

import ast
import csv
import gc
import hashlib
import json
import math
import os
import random
import re
import shutil
import subprocess
import sys
import time
import zipfile
from pathlib import Path

import librosa
import numpy as np
import pandas as pd
import requests
import soundfile as sf
import torch
import torchaudio
from scipy.signal import butter, sosfilt
from sklearn.metrics import roc_auc_score, roc_curve
from tqdm.auto import tqdm

SEED = 20260903
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DRIVE = Path("/content/drive/MyDrive")
PROJECT = DRIVE / "deepvoice_ensemble_ood2500"
PROJECT.mkdir(parents=True, exist_ok=True)

# DACON에서 직접 받은 파일
BASELINE_OR_OPEN_ZIP = DRIVE / "dacon_baseline" / "open.zip"

# 마지막 SONICS DynamicMix 학습 노트북의 기본 출력
AASIST_BEST_PT = (
    DRIVE / "deepvoice_dynamic25k" / "runs" /
    "exp03_aasist_rawboost" / "best.pt"
)

LOCAL = Path("/content/deepvoice_ensemble")
ARCHIVES = PROJECT / "archives"       # 재실행을 위해 Drive에 보존; 속도 우선이면 /content로 변경
RAW = LOCAL / "raw"
STAGE = LOCAL / "submit_stage"
OOD_ROOT = LOCAL / "ood2500"
OUTPUT_ZIP = PROJECT / "submit.zip"
OOD_PREDICTIONS = PROJECT / "ood2500_predictions.csv"
OOD_REPORT = PROJECT / "ood2500_official_metrics.json"

# 요청한 완전 별도 음성 OOD. 자동 대안은 in_the_wild_auto.
VOICE_SOURCE_MODE = "speechfake_manual"  # 또는 "in_the_wild_auto"
SPEECHFAKE_REAL_DIR = DRIVE / "ood_sources" / "speechfake" / "real_test"
SPEECHFAKE_FAKE_DIR = DRIVE / "ood_sources" / "speechfake" / "fake_test"

POOL_SIZE = 625
OOD_SIZE = 2_500
SAMPLE_RATE = 16_000
MIN_SECONDS = 4
MAX_SECONDS = 60

# 1,200개 DACON inference의 시간 예산. 값은 package script에도 삽입됩니다.
MAX_SEPARATION_SECONDS = 24
MAX_DF_SEGMENTS = 3
MAX_AASIST_SEGMENTS = 4
BENCHMARK_FILES = 80
TARGET_TEST_FILES = 1_200
TIME_LIMIT_MINUTES = 50.0
TIME_SAFETY_FACTOR = 1.15
ENFORCE_50_MIN_GATE = True

# OOD를 test로 유지하기 위해 아래 weight는 고정합니다.
AASIST_COMPONENT_WEIGHT = 0.30
AASIST_PRESENCE_WEIGHT = 0.20
AASIST_FILE_WEIGHT = 0.35

DOWNLOAD_OOD_ARCHIVES = True
BUILD_OOD_DATASET = True
RUN_OOD_EVALUATION = True
RUN_L4_BENCHMARK = True
BUILD_SUBMIT_ZIP = True

for directory in (ARCHIVES, RAW, LOCAL):
    directory.mkdir(parents=True, exist_ok=True)

print("torch:", torch.__version__)
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("free /content GiB:", round(shutil.disk_usage("/content").free / 2**30, 1))
print("baseline:", BASELINE_OR_OPEN_ZIP)
print("AASIST:", AASIST_BEST_PT)


## 2. 안전한 다운로드·압축 해제·무결성 도우미


In [ ]:
AUDIO_EXTENSIONS = {".wav", ".flac", ".mp3", ".m4a", ".aac", ".ogg", ".opus", ".wma"}
PREDICTION_COLUMNS = [
    "FILE_FAKE_PROB", "VOICE_FAKE_PROB", "MUSIC_FAKE_PROB",
    "VOICE_PRESENT_PROB", "MUSIC_PRESENT_PROB",
]
TRUTH_COLUMNS = ["FILE_FAKE", "VOICE_FAKE", "MUSIC_FAKE", "VOICE_PRESENT", "MUSIC_PRESENT"]


def sha256(path: Path, chunk_size: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as file:
        while chunk := file.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


def safe_extract(zip_path: Path, destination: Path) -> None:
    destination = destination.resolve()
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if destination not in target.parents and target != destination:
                raise ValueError(f"unsafe ZIP member: {member.filename}")
        archive.extractall(destination)


def download_resumable(url: str, destination: Path) -> Path:
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(destination.suffix + ".part")
    existing = temporary.stat().st_size if temporary.exists() else 0
    headers = {"Range": f"bytes={existing}-"} if existing else {}
    mode = "ab" if existing else "wb"
    with requests.get(url, headers=headers, stream=True, timeout=(30, 600), allow_redirects=True) as response:
        if existing and response.status_code == 200:
            existing, mode = 0, "wb"
        response.raise_for_status()
        total = int(response.headers.get("content-length", 0)) + existing
        with temporary.open(mode) as file, tqdm(
            total=total, initial=existing, unit="B", unit_scale=True, desc=destination.name
        ) as progress:
            for chunk in response.iter_content(chunk_size=8 * 1024 * 1024):
                if chunk:
                    file.write(chunk)
                    progress.update(len(chunk))
    temporary.replace(destination)
    return destination


def find_audio(root: Path) -> list[Path]:
    return sorted(
        p for p in root.rglob("*")
        if p.is_file() and p.suffix.lower() in AUDIO_EXTENSIONS
    )


def find_baseline_zip(source: Path, extraction_root: Path) -> Path:
    if not source.is_file():
        raise FileNotFoundError(
            f"{source} 파일이 없습니다. DACON 데이터 탭의 open.zip 또는 공식 baseline_submit.zip을 Drive에 올리세요."
        )
    if source.name.lower() in {"baseline_submit.zip", "submit.zip"}:
        return source
    safe_extract(source, extraction_root)
    matches = list(extraction_root.rglob("baseline_submit.zip"))
    if len(matches) != 1:
        raise FileNotFoundError(f"baseline_submit.zip을 정확히 하나 찾아야 합니다: {matches}")
    return matches[0]


def directory_size(root: Path) -> int:
    return sum(p.stat().st_size for p in root.rglob("*") if p.is_file())


print("helpers: OK")


## 3. 공식 baseline과 AASIST `best.pt` 검사

checkpoint가 실제 AASIST 5-head 모델인지, 공식 다섯 출력 순서가 맞는지 먼저 확인합니다.


In [ ]:
if not AASIST_BEST_PT.is_file():
    raise FileNotFoundError(f"AASIST best.pt가 없습니다: {AASIST_BEST_PT}")

checkpoint = torch.load(AASIST_BEST_PT, map_location="cpu", weights_only=False)
config = checkpoint.get("config", {})
if config.get("model") != "aasist":
    raise ValueError(f"exp03 AASIST checkpoint가 아닙니다: {config}")
if "model_state" not in checkpoint:
    raise KeyError("checkpoint에 model_state가 없습니다")
if checkpoint.get("dacon_columns") not in (None, PREDICTION_COLUMNS):
    raise ValueError(f"checkpoint output 순서가 다릅니다: {checkpoint.get('dacon_columns')}")
if not any(key.startswith("net.") for key in checkpoint["model_state"]):
    raise ValueError("AASIST net.* weight를 찾지 못했습니다")

baseline_extract = LOCAL / "baseline_open"
if baseline_extract.exists():
    shutil.rmtree(baseline_extract)
baseline_zip = find_baseline_zip(BASELINE_OR_OPEN_ZIP, baseline_extract)

if STAGE.exists():
    shutil.rmtree(STAGE)
safe_extract(baseline_zip, STAGE)
for required in (STAGE / "model", STAGE / "script.py", STAGE / "requirements.txt"):
    if not required.exists():
        raise FileNotFoundError(f"공식 baseline 필수 항목 누락: {required}")

baseline_script = (STAGE / "script.py").read_text(encoding="utf-8")
ast.parse(baseline_script)
for marker in ("DF_ARENA_DIR", "load_df_arena_model", "separate_voice_and_music", "submission.csv"):
    if marker not in baseline_script:
        raise RuntimeError(f"예상한 공식 baseline marker가 없습니다: {marker}")

print("checkpoint epoch:", checkpoint.get("epoch"))
print("validation:", checkpoint.get("validation_report"))
print("baseline zip:", baseline_zip)
print("baseline extracted GiB:", round(directory_size(STAGE) / 2**30, 2))


## 4. 학습 당시와 동일한 AASIST 소스 준비

checkpoint에 기록된 git commit을 우선 사용합니다. 평가 서버는 추론 중 인터넷이 차단되므로 코드와 config를 ZIP 안에 포함합니다.


In [ ]:
def run(command, cwd=None):
    print("+", " ".join(map(str, command)))
    subprocess.run(command, cwd=cwd, check=True)


repo_commit = (checkpoint.get("repo_commits") or {}).get("aasist")
repo_root = LOCAL / "aasist_repo"
if repo_root.exists():
    shutil.rmtree(repo_root)
run(["git", "clone", "https://github.com/clovaai/aasist.git", str(repo_root)])
if repo_commit:
    run(["git", "checkout", repo_commit], cwd=repo_root)
actual_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=repo_root, text=True
).strip()
if repo_commit and actual_commit != repo_commit:
    raise RuntimeError((repo_commit, actual_commit))

aasist_asset = STAGE / "model" / "aasist_ensemble"
aasist_asset.mkdir(parents=True, exist_ok=True)
shutil.copy2(AASIST_BEST_PT, aasist_asset / "best.pt")
shutil.copytree(repo_root / "models", aasist_asset / "code" / "models")
shutil.copytree(repo_root / "config", aasist_asset / "code" / "config")
for optional in ("LICENSE", "README.md"):
    if (repo_root / optional).is_file():
        shutil.copy2(repo_root / optional, aasist_asset / optional)

print("AASIST commit:", actual_commit)
print("AASIST checkpoint SHA256:", sha256(aasist_asset / "best.pt"))


## 5. 공식 `script.py`에 AASIST ensemble과 시간 예산 추가

- DF-Arena fake score 70% + AASIST component score 30%
- PANNs presence 80% + AASIST presence 20%
- 구조적 file score 65% + AASIST direct file score 35%

확률 평균보다 극단값이 덜 무너지는 **logit 평균**을 사용합니다. AASIST는 원본 오디오에서 최대 4개 segment만 보므로 추가 시간은 작습니다.


In [ ]:
script_path = STAGE / "script.py"
script = script_path.read_text(encoding="utf-8")

# 모델 경로 추가
path_marker = 'DF_ARENA_DIR = MODEL_DIR / "df_arena_1b"'
if path_marker not in script:
    raise RuntimeError("공식 baseline의 DF_ARENA_DIR marker가 없습니다")
script = script.replace(
    path_marker,
    path_marker + '\nAASIST_DIR = MODEL_DIR / "aasist_ensemble"\nAASIST_CODE_DIR = AASIST_DIR / "code"',
    1,
)

# 제출 서버 시간 예산 상수
constant_marker = "SILENCE_RMS = 1e-5"
budget_constants = f'''{constant_marker}
MAX_SEPARATION_SECONDS = {MAX_SEPARATION_SECONDS}
MAX_DF_SEGMENTS = {MAX_DF_SEGMENTS}
MAX_AASIST_SEGMENTS = {MAX_AASIST_SEGMENTS}
AASIST_COMPONENT_WEIGHT = {AASIST_COMPONENT_WEIGHT}
AASIST_PRESENCE_WEIGHT = {AASIST_PRESENCE_WEIGHT}
AASIST_FILE_WEIGHT = {AASIST_FILE_WEIGHT}'''
if constant_marker not in script:
    raise RuntimeError("공식 baseline의 SILENCE_RMS marker가 없습니다")
script = script.replace(constant_marker, budget_constants, 1)

# HTDemucs 입력을 start/middle/end의 총 24초로 제한. 긴 파일의 특정 위치만 보는 편향을 줄인다.
load_track_marker = "waveform = load_track(audio_path, model.audio_channels, model.samplerate).float()"
if load_track_marker not in script:
    raise RuntimeError("공식 baseline의 load_track marker가 없습니다")
script = script.replace(
    load_track_marker,
    load_track_marker + "\n    waveform = budget_waveform(waveform, model.samplerate, MAX_SEPARATION_SECONDS)",
    1,
)

# DF-Arena section에서만 segment 수를 제한한다. PANNs presence는 원본 baseline coverage를 유지한다.
df_pattern = re.compile(
    r"# -+\n# 5\. DF-Arena 1B.*?(?=# -+\n# 6\.)",
    flags=re.DOTALL,
)
df_match = df_pattern.search(script)
if not df_match:
    raise RuntimeError("공식 baseline의 DF-Arena section을 찾지 못했습니다")
df_section = df_match.group(0)
if "get_segment_starts(audio.size)" not in df_section:
    raise RuntimeError("DF-Arena segment loop marker가 없습니다")
df_section = df_section.replace(
    "get_segment_starts(audio.size)",
    "get_budgeted_segment_starts(audio.size, MAX_DF_SEGMENTS)",
)
script = script[:df_match.start()] + df_section + script[df_match.end():]

aasist_injection = r'''

# -----------------------------------------------------------------------------
# 5B. User-trained AASIST five-head ensemble and deterministic time budget
# -----------------------------------------------------------------------------

def budget_waveform(waveform, sample_rate, max_seconds):
    max_samples = max(1, int(sample_rate * max_seconds))
    if waveform.shape[-1] <= max_samples:
        return waveform
    piece = max_samples // 3
    starts = [0, (waveform.shape[-1] - piece) // 2, waveform.shape[-1] - piece]
    return torch.cat([waveform[..., start:start + piece] for start in starts], dim=-1)


def get_budgeted_segment_starts(audio_length, maximum_segments):
    starts = get_segment_starts(audio_length)
    if len(starts) <= maximum_segments:
        return starts
    indices = np.linspace(0, len(starts) - 1, maximum_segments).round().astype(int)
    return [starts[index] for index in sorted(set(indices.tolist()))]


class AASISTFiveHead(torch.nn.Module):
    def __init__(self, model_config):
        super().__init__()
        if str(AASIST_CODE_DIR) not in sys.path:
            sys.path.insert(0, str(AASIST_CODE_DIR))
        from models.AASIST import Model as OfficialAASIST
        self.net = OfficialAASIST(model_config)
        self.net.out_layer = torch.nn.Linear(self.net.out_layer.in_features, 5)

    def forward(self, audio):
        _, logits = self.net(audio, Freq_aug=False)
        return logits


def load_aasist_model(device):
    with (AASIST_CODE_DIR / "config" / "AASIST.conf").open(encoding="utf-8") as file:
        model_config = json.load(file)["model_config"]
    checkpoint = torch.load(AASIST_DIR / "best.pt", map_location="cpu", weights_only=False)
    if checkpoint.get("config", {}).get("model") != "aasist":
        raise ValueError("Packaged best.pt is not an AASIST checkpoint")
    model = AASISTFiveHead(model_config)
    model.load_state_dict(checkpoint["model_state"], strict=True)
    return model.to(device).eval()


def canonicalize_aasist_segment(segment):
    segment = np.asarray(segment, dtype=np.float32)
    segment = np.nan_to_num(segment) - float(np.mean(segment))
    rms = float(np.sqrt(np.mean(segment.astype(np.float64) ** 2)))
    if rms < 1e-7:
        return np.zeros_like(segment)
    peak = max(float(np.max(np.abs(segment))), 1e-8)
    return np.clip(segment / peak * 0.82, -1, 1).astype(np.float32)


def predict_aasist(model, audio, device):
    starts = get_budgeted_segment_starts(audio.size, MAX_AASIST_SEGMENTS)
    segments = np.stack([
        canonicalize_aasist_segment(extract_segment(audio, start)) for start in starts
    ])
    tensors = torch.from_numpy(segments)
    outputs = []
    with torch.inference_mode():
        for start in range(0, len(tensors), 16):
            logits = model(tensors[start:start + 16].to(device, non_blocking=True))
            outputs.append(torch.sigmoid(logits.float()).cpu().numpy())
    matrix = np.concatenate(outputs)
    fake = 0.6 * matrix[:, :3].max(0) + 0.4 * matrix[:, :3].mean(0)
    presence = 0.7 * matrix[:, 3:].max(0) + 0.3 * matrix[:, 3:].mean(0)
    return np.concatenate([fake, presence]).astype(float)


def logit_blend(base_probability, aasist_probability, aasist_weight):
    epsilon = 1e-5
    base = float(np.clip(base_probability, epsilon, 1 - epsilon))
    aasist = float(np.clip(aasist_probability, epsilon, 1 - epsilon))
    base_logit = np.log(base / (1 - base))
    aasist_logit = np.log(aasist / (1 - aasist))
    value = (1 - aasist_weight) * base_logit + aasist_weight * aasist_logit
    return float(1 / (1 + np.exp(-value)))

'''

section6 = re.search(r"(?m)^# -+\n# 6\.", script)
if not section6:
    raise RuntimeError("공식 baseline section 6 marker가 없습니다")
script = script[:section6.start()] + aasist_injection + "\n" + script[section6.start():]

# 모델 로드와 file loop에 AASIST를 추가한다.
load_marker = "df_arena_model, fake_label_index = load_df_arena_model(device)"
if load_marker not in script:
    raise RuntimeError("DF-Arena load call marker가 없습니다")
script = script.replace(
    load_marker,
    load_marker + "\n    aasist_model = load_aasist_model(device)",
    1,
)

music_call = '''        music_fake = predict_fake(
            df_arena_model, fake_label_index, music_audio, device
        )'''
if music_call not in script:
    raise RuntimeError("DF-Arena music predict call marker가 없습니다")
script = script.replace(
    music_call,
    music_call + '''
        aasist_scores = predict_aasist(
            aasist_model, load_audio(audio_path), device
        )
        voice_fake = logit_blend(
            voice_fake, aasist_scores[1], AASIST_COMPONENT_WEIGHT
        )
        music_fake = logit_blend(
            music_fake, aasist_scores[2], AASIST_COMPONENT_WEIGHT
        )''',
    1,
)

presence_marker = "voice_present, music_present = presence_scores[audio_path.stem]"
if presence_marker not in script:
    raise RuntimeError("presence fusion marker가 없습니다")
script = script.replace(
    presence_marker,
    '''voice_present_base, music_present_base = presence_scores[audio_path.stem]
        voice_present = logit_blend(
            voice_present_base, aasist_scores[3], AASIST_PRESENCE_WEIGHT
        )
        music_present = logit_blend(
            music_present_base, aasist_scores[4], AASIST_PRESENCE_WEIGHT
        )''',
    1,
)

file_marker = '''file_fake = combine_file_fake_score(
            voice_fake, music_fake, voice_present, music_present
        )'''
if file_marker not in script:
    raise RuntimeError("file fusion marker가 없습니다")
script = script.replace(
    file_marker,
    '''component_file_fake = combine_file_fake_score(
            voice_fake, music_fake, voice_present, music_present
        )
        file_fake = logit_blend(
            component_file_fake, aasist_scores[0], AASIST_FILE_WEIGHT
        )''',
    1,
)

script_path.write_text(script, encoding="utf-8")
ast.parse(script)

required_markers = [
    "load_df_arena_model", "load_htdemucs_model", "load_aasist_model",
    "predict_aasist", "logit_blend", "MAX_DF_SEGMENTS",
    'Path("output") / "submission.csv"',
]
missing = [marker for marker in required_markers if marker not in script]
if missing:
    raise RuntimeError(f"patched script marker 누락: {missing}")

manifest = {
    "architecture": "Official PANNs + budgeted HTDemucs + DF-Arena 1B + AASIST five-head",
    "aasist_commit": actual_commit,
    "aasist_checkpoint_sha256": sha256(aasist_asset / "best.pt"),
    "time_budget": {
        "max_separation_seconds": MAX_SEPARATION_SECONDS,
        "max_df_segments": MAX_DF_SEGMENTS,
        "max_aasist_segments": MAX_AASIST_SEGMENTS,
    },
    "weights": {
        "aasist_component": AASIST_COMPONENT_WEIGHT,
        "aasist_presence": AASIST_PRESENCE_WEIGHT,
        "aasist_file": AASIST_FILE_WEIGHT,
    },
}
(aasist_asset / "ENSEMBLE_MANIFEST.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("patched script.py AST: OK")


## 6. OOD 원본 데이터 다운로드

엄격 OOD 기본 구성:

- REAL/FAKE Voice: [SpeechFake](https://huggingface.co/datasets/DeepFense/SpeechFake) test에서 미리 선별한 두 폴더
- REAL Music: [Song Describer](https://zenodo.org/records/10072001)
- FAKE Music: [FakeMusicCaps](https://zenodo.org/records/15063698)

SpeechFake 전체 archive는 매우 커서 Colab에서 자동으로 전부 받지 않습니다. `SPEECHFAKE_REAL_DIR`, `SPEECHFAKE_FAKE_DIR`에 test audio를 두면 각각 625개를 읽습니다. 간편 자동 대안이 필요할 때만 `in_the_wild_auto`로 바꾸면 [In-the-Wild](https://huggingface.co/datasets/mueller91/In-The-Wild) 단일 ZIP을 받습니다.


In [ ]:
URLS = {
    "in_the_wild": "https://huggingface.co/datasets/mueller91/In-The-Wild/resolve/main/release_in_the_wild.zip",
    "song_describer_audio": "https://zenodo.org/api/records/10072001/files/audio.zip/content",
    "song_describer_csv": "https://zenodo.org/api/records/10072001/files/song_describer.csv/content",
    "fake_music_caps": "https://zenodo.org/api/records/15063698/files/FakeMusicCaps.zip/content",
    "musiccaps_csv": "https://huggingface.co/datasets/google/MusicCaps/resolve/main/musiccaps-public.csv",
}

archive_paths = {
    "in_the_wild": ARCHIVES / "release_in_the_wild.zip",
    "song_describer_audio": ARCHIVES / "song_describer_audio.zip",
    "song_describer_csv": ARCHIVES / "song_describer.csv",
    "fake_music_caps": ARCHIVES / "FakeMusicCaps.zip",
    "musiccaps_csv": ARCHIVES / "musiccaps-public.csv",
}

if DOWNLOAD_OOD_ARCHIVES:
    needed = ["song_describer_audio", "song_describer_csv", "fake_music_caps", "musiccaps_csv"]
    if VOICE_SOURCE_MODE == "in_the_wild_auto":
        needed.insert(0, "in_the_wild")
    for key in needed:
        if not archive_paths[key].is_file():
            download_resumable(URLS[key], archive_paths[key])
        print(key, round(archive_paths[key].stat().st_size / 2**30, 3), "GiB")

extract_targets = {
    "in_the_wild": RAW / "in_the_wild",
    "song_describer_audio": RAW / "song_describer",
    "fake_music_caps": RAW / "fake_music_caps",
}
for key, target in extract_targets.items():
    if key == "in_the_wild" and VOICE_SOURCE_MODE != "in_the_wild_auto":
        continue
    marker = target / ".extracted.ok"
    if not marker.exists():
        target.mkdir(parents=True, exist_ok=True)
        safe_extract(archive_paths[key], target)
        marker.touch()
    print(key, "audio files:", len(find_audio(target)))


## 7. 네 OOD source pool을 각각 정확히 625개 선택

음성은 label과 speaker를, FakeMusicCaps는 생성기 폴더명을, Song Describer는 artist ID를 기준으로 가능한 한 균형 선택합니다. 최종 source manifest를 Drive에 저장합니다.


In [ ]:
def normalize_binary_label(value) -> str | None:
    value = str(value).strip().lower().replace("_", "-")
    if value in {"real", "bonafide", "bona-fide", "genuine", "0"}:
        return "real"
    if value in {"fake", "spoof", "deepfake", "1"}:
        return "fake"
    return None


def index_labeled_voice(root: Path) -> pd.DataFrame:
    audio_files = find_audio(root)
    by_name = {p.name: p for p in audio_files}
    by_stem = {p.stem: p for p in audio_files}
    rows = []
    for table_path in sorted(list(root.rglob("*.csv")) + list(root.rglob("*.tsv"))):
        try:
            table = pd.read_csv(table_path, sep="\t" if table_path.suffix == ".tsv" else ",")
        except Exception:
            continue
        lowered = {str(c).lower(): c for c in table.columns}
        label_col = next((lowered[k] for k in ("label", "class", "target") if k in lowered), None)
        file_col = next((lowered[k] for k in ("file", "path", "filename", "audio") if k in lowered), None)
        if label_col is None or file_col is None:
            continue
        speaker_col = next((lowered[k] for k in ("speaker", "speaker_id", "person") if k in lowered), None)
        for _, row in table.iterrows():
            label = normalize_binary_label(row[label_col])
            raw_name = str(row[file_col])
            path = by_name.get(Path(raw_name).name) or by_stem.get(Path(raw_name).stem)
            if label and path:
                rows.append({
                    "path": str(path), "label": label,
                    "group": str(row[speaker_col]) if speaker_col else path.parent.name,
                })
        if rows:
            break
    if not rows:
        for path in audio_files:
            tokens = " ".join(part.lower() for part in path.parts)
            label = "fake" if any(t in tokens for t in ("fake", "spoof")) else (
                "real" if any(t in tokens for t in ("real", "bonafide", "bona-fide")) else None
            )
            if label:
                rows.append({"path": str(path), "label": label, "group": path.parent.name})
    frame = pd.DataFrame(rows).drop_duplicates("path")
    if frame.empty:
        raise RuntimeError(f"voice label metadata를 찾지 못했습니다: {root}")
    return frame


def deterministic_group_sample(frame: pd.DataFrame, count: int, seed_key: str) -> pd.DataFrame:
    work = frame.copy()
    work["_key"] = work["path"].map(
        lambda value: hashlib.sha256(f"{SEED}|{seed_key}|{value}".encode()).hexdigest()
    )
    groups = [g.sort_values("_key").to_dict("records") for _, g in work.groupby("group", sort=True)]
    selected, index = [], 0
    while len(selected) < count and any(index < len(group) for group in groups):
        for group in groups:
            if index < len(group) and len(selected) < count:
                selected.append(group[index])
        index += 1
    if len(selected) != count:
        raise ValueError(f"{seed_key}: {count}개 필요, 사용 가능 {len(selected)}개")
    return pd.DataFrame(selected).drop(columns=["_key"], errors="ignore")


if VOICE_SOURCE_MODE == "in_the_wild_auto":
    voice_index = index_labeled_voice(RAW / "in_the_wild")
else:
    real_paths = find_audio(SPEECHFAKE_REAL_DIR)
    fake_paths = find_audio(SPEECHFAKE_FAKE_DIR)
    voice_index = pd.DataFrame(
        [{"path": str(p), "label": "real", "group": p.parent.name} for p in real_paths] +
        [{"path": str(p), "label": "fake", "group": p.parent.name} for p in fake_paths]
    )

real_voice = deterministic_group_sample(voice_index[voice_index.label == "real"], POOL_SIZE, "real_voice")
fake_voice = deterministic_group_sample(voice_index[voice_index.label == "fake"], POOL_SIZE, "fake_voice")

# Song Describer: caption table에서 track/artist/path를 복원
song_table = pd.read_csv(archive_paths["song_describer_csv"])
song_table = song_table.sort_values("caption_id").groupby("track_id", as_index=False).agg({
    "path": "first", "artist_id": "first", "caption": " ".join,
})
song_audio = find_audio(RAW / "song_describer")
song_lookup = {p.as_posix().lower().split("/audio/")[-1]: p for p in song_audio}
song_lookup.update({p.name.lower(): p for p in song_audio})
song_rows = []
for _, row in song_table.iterrows():
    rel = str(row["path"]).replace("\\", "/").lower()
    path = song_lookup.get(rel) or song_lookup.get(Path(rel).name.lower())
    if path:
        song_rows.append({
            "path": str(path), "group": str(row["artist_id"]),
            "caption": str(row["caption"]),
        })
real_music_index = pd.DataFrame(song_rows).drop_duplicates("path")
real_music = deterministic_group_sample(real_music_index, POOL_SIZE, "real_music")


def infer_generator(path: Path) -> str:
    compact = re.sub(r"[^a-z0-9]", "", path.as_posix().lower())
    aliases = {
        "musicgen": "MusicGen", "musicldm": "MusicLDM", "audioldm2": "AudioLDM2",
        "stableaudioopen": "StableAudioOpen", "mustango": "Mustango",
    }
    for token, name in aliases.items():
        if token in compact:
            return name
    return "unknown"


musiccaps_table = pd.read_csv(archive_paths["musiccaps_csv"])
musiccaps_caption = dict(zip(
    musiccaps_table["ytid"].astype(str), musiccaps_table["caption"].fillna("").astype(str)
))
fake_music_rows = []
for path in find_audio(RAW / "fake_music_caps"):
    generator = infer_generator(path)
    if generator != "unknown":
        fake_music_rows.append({
            "path": str(path), "group": generator, "generator": generator,
            "caption": musiccaps_caption.get(path.stem, ""),
        })
fake_music_index = pd.DataFrame(fake_music_rows).drop_duplicates("path")
expected_generators = {"MusicGen", "MusicLDM", "AudioLDM2", "StableAudioOpen", "Mustango"}
available_generators = set(fake_music_index.get("generator", []))
if available_generators != expected_generators:
    raise RuntimeError(f"FakeMusicCaps generator 폴더 인식 실패: {sorted(available_generators)}")

parts = []
for generator in sorted(expected_generators):
    subset = fake_music_index[fake_music_index.generator == generator]
    parts.append(deterministic_group_sample(subset, 125, f"fake_music_{generator}"))
fake_music = pd.concat(parts, ignore_index=True)

pools = {}
for pool_name, frame in {
    "real_voice": real_voice, "fake_voice": fake_voice,
    "real_music": real_music, "fake_music": fake_music,
}.items():
    frame = frame.copy()
    frame["pool"] = pool_name
    frame["source_id"] = [f"{pool_name}_{i:04d}" for i in range(len(frame))]
    pools[pool_name] = frame
    assert len(frame) == POOL_SIZE

source_manifest = pd.concat(pools.values(), ignore_index=True)
source_manifest.to_csv(PROJECT / "ood_source_manifest_2500.csv", index=False)
display(source_manifest.groupby(["pool", "group"]).size().groupby(level=0).agg(["count", "min", "max", "sum"]))
print("unique source files:", source_manifest.path.nunique())
assert len(source_manifest) == 4 * POOL_SIZE == OOD_SIZE


## 8. DACON 조건형 고정 OOD DynamicMix 2,500개 생성

무작위 augmentation이 아니라 seed와 manifest가 고정된 **평가 데이터 생성**입니다.

- 길이: 4~60초
- 최종 sample rate: 모두 16 kHz
- 채널: mono/stereo 혼합
- 형식: WAV/FLAC/MP3/AAC/Opus
- 일부: 8 kHz μ-law와 300~3400 Hz band를 거친 뒤 16 kHz로 복원한 전화채널
- 8가지 REAL/FAKE Voice/Music 조합을 312~313개씩 균등 생성

음악에 원래 포함된 보컬은 caption keyword로 기록합니다. 완벽한 수동 annotation이 아니므로 `contains_voice_hint`와 manifest를 함께 보관해야 합니다.


In [ ]:
RECIPE_COMPONENTS = {
    "rv": ["real_voice"], "fv": ["fake_voice"],
    "rm": ["real_music"], "fm": ["fake_music"],
    "rv_rm": ["real_voice", "real_music"],
    "fv_rm": ["fake_voice", "real_music"],
    "rv_fm": ["real_voice", "fake_music"],
    "fv_fm": ["fake_voice", "fake_music"],
}

VOICE_WORDS = re.compile(
    r"\b(vocal|vocals|voice|voices|singer|singing|sung|lyrics|spoken|speech|whisper|rap|rapper|choir|chant)\b",
    re.IGNORECASE,
)


def read_audio_16k(path: Path) -> np.ndarray:
    try:
        audio, sr = sf.read(path, dtype="float32", always_2d=True)
        audio = audio.mean(axis=1)
    except Exception:
        audio, sr = librosa.load(path, sr=None, mono=True, dtype=np.float32)
    if len(audio) == 0 or not np.isfinite(audio).all():
        raise ValueError(f"invalid audio: {path}")
    if sr != SAMPLE_RATE:
        audio = librosa.resample(audio, orig_sr=sr, target_sr=SAMPLE_RATE, res_type="soxr_hq")
    return np.asarray(audio, dtype=np.float32)


def crop_or_tile(audio: np.ndarray, samples: int, rng: np.random.Generator) -> np.ndarray:
    if len(audio) >= samples:
        start = int(rng.integers(0, len(audio) - samples + 1))
        return audio[start:start + samples].copy()
    repeats = math.ceil(samples / len(audio))
    tiled = np.tile(audio, repeats)[:samples].copy()
    # 반복 경계의 click를 줄이기 위한 짧은 fade
    fade = min(320, len(audio) // 8)
    if fade > 1:
        for boundary in range(len(audio), samples, len(audio)):
            left, right = max(0, boundary - fade), min(samples, boundary + fade)
            if right - left == 2 * fade:
                tiled[left:boundary] *= np.linspace(1, 0, fade, endpoint=False)
                tiled[boundary:right] *= np.linspace(0, 1, fade, endpoint=False)
    return tiled


def rms_to_db(audio: np.ndarray) -> float:
    return float(20 * np.log10(np.sqrt(np.mean(audio.astype(np.float64) ** 2)) + 1e-9))


def set_rms(audio: np.ndarray, target_db: float) -> np.ndarray:
    gain = 10 ** ((target_db - rms_to_db(audio)) / 20)
    return (audio * gain).astype(np.float32)


def telephone_channel(audio: np.ndarray) -> np.ndarray:
    down = librosa.resample(audio, orig_sr=16_000, target_sr=8_000, res_type="soxr_hq")
    sos = butter(6, [300, 3400], btype="bandpass", fs=8_000, output="sos")
    down = sosfilt(sos, down).astype(np.float32)
    tensor = torch.from_numpy(np.clip(down, -1, 1))
    encoded = torchaudio.functional.mu_law_encoding(tensor, 256)
    decoded = torchaudio.functional.mu_law_decoding(encoded, 256).numpy()
    return librosa.resample(decoded, orig_sr=8_000, target_sr=16_000, res_type="soxr_hq").astype(np.float32)


def make_stereo(audio: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    delay = int(rng.integers(2, 33))
    right = np.roll(audio, delay) * float(rng.uniform(0.88, 1.0))
    right[:delay] = 0
    return np.stack([audio, right], axis=1).astype(np.float32)


CODEC_PROFILES = [
    ("wav", None), ("flac", None),
    ("mp3", "64k"), ("mp3", "96k"), ("mp3", "128k"), ("mp3", "192k"),
    ("aac", "64k"), ("aac", "96k"), ("aac", "128k"),
    ("opus", "48k"), ("opus", "64k"), ("opus", "96k"),
]


def encode_audio(audio: np.ndarray, destination: Path, codec: str, bitrate: str | None) -> None:
    temporary = destination.with_suffix(".source.wav")
    sf.write(temporary, audio, SAMPLE_RATE, subtype="PCM_16")
    codec_args = {
        "wav": ["-c:a", "pcm_s16le"],
        "flac": ["-c:a", "flac"],
        "mp3": ["-c:a", "libmp3lame", "-b:a", bitrate],
        "aac": ["-c:a", "aac", "-b:a", bitrate],
        "opus": ["-c:a", "libopus", "-b:a", bitrate],
    }[codec]
    command = [
        "ffmpeg", "-hide_banner", "-loglevel", "error", "-y", "-i", str(temporary),
        "-ar", str(SAMPLE_RATE), *codec_args, str(destination),
    ]
    subprocess.run(command, check=True)
    temporary.unlink()


def duration_schedule(count: int, rng: np.random.Generator) -> list[float]:
    anchors = np.array([4, 5, 6, 8, 10, 12, 15, 20, 30, 45, 60], dtype=float)
    weights = np.array([8, 8, 10, 12, 14, 12, 10, 8, 5, 2, 1], dtype=float)
    values = rng.choice(anchors, size=count, p=weights / weights.sum())
    jitter = rng.uniform(-0.35, 0.35, size=count)
    return np.clip(values + jitter, MIN_SECONDS, MAX_SECONDS).round(3).tolist()


def fixed_recipe_types() -> list[str]:
    names = list(RECIPE_COMPONENTS)
    counts = [313 if index < 4 else 312 for index in range(len(names))]
    types = [name for name, count in zip(names, counts) for _ in range(count)]
    random.Random(SEED).shuffle(types)
    assert len(types) == OOD_SIZE
    return types


def contains_voice_hint(record: dict) -> bool:
    if str(record.get("pool", "")).endswith("voice"):
        return True
    return bool(VOICE_WORDS.search(str(record.get("caption", ""))))


if BUILD_OOD_DATASET:
    if OOD_ROOT.exists():
        shutil.rmtree(OOD_ROOT)
    test_dir = OOD_ROOT / "data" / "test"
    test_dir.mkdir(parents=True)
    rng = np.random.default_rng(SEED)
    durations = duration_schedule(OOD_SIZE, rng)
    recipes = fixed_recipe_types()
    pool_records = {
        name: frame.sample(frac=1, random_state=SEED).to_dict("records")
        for name, frame in pools.items()
    }
    counters = {name: 0 for name in pool_records}
    rows, truth_rows = [], []

    for index, (recipe_type, duration) in enumerate(tqdm(zip(recipes, durations), total=OOD_SIZE)):
        identifier = f"OOD_{index:05d}"
        components, selected_records = [], []
        samples = int(round(duration * SAMPLE_RATE))
        for pool_name in RECIPE_COMPONENTS[recipe_type]:
            records = pool_records[pool_name]
            record = records[counters[pool_name] % len(records)]
            counters[pool_name] += 1
            audio = crop_or_tile(read_audio_16k(Path(record["path"])), samples, rng)
            audio = set_rms(audio - audio.mean(), float(rng.uniform(-25, -19)))
            components.append(audio)
            selected_records.append(record)

        if len(components) == 2:
            snr = float(rng.uniform(-10, 10))
            components[0] = set_rms(components[0], -22 + snr / 2)
            components[1] = set_rms(components[1], -22 - snr / 2)
            layout = rng.choice(["overlap", "overlap", "partial", "sequential"])
            if layout == "partial":
                for component_index in range(2):
                    active = int(rng.integers(max(1, SAMPLE_RATE), samples + 1))
                    start = int(rng.integers(0, samples - active + 1))
                    mask = np.zeros(samples, dtype=np.float32)
                    mask[start:start + active] = 1
                    components[component_index] *= mask
            elif layout == "sequential":
                boundary = int(rng.integers(int(samples * .35), int(samples * .65)))
                components[0][boundary:] = 0
                components[1][:boundary] = 0
        else:
            layout = "single"

        mixed = np.sum(components, axis=0).astype(np.float32)
        phone = bool(rng.random() < 0.12)
        if phone:
            mixed = telephone_channel(mixed)
            mixed = crop_or_tile(mixed, samples, rng)
        mixed = set_rms(mixed - mixed.mean(), float(rng.uniform(-24, -18)))
        peak = max(float(np.max(np.abs(mixed))), 1e-8)
        mixed = np.clip(mixed / peak * float(rng.uniform(.72, .96)), -1, 1)

        stereo = bool(rng.random() < 0.30)
        if stereo:
            mixed = make_stereo(mixed, rng)
        codec, bitrate = CODEC_PROFILES[index % len(CODEC_PROFILES)]
        extension = "m4a" if codec == "aac" else codec
        destination = test_dir / f"{identifier}.{extension}"
        encode_audio(mixed, destination, codec, bitrate)

        voice_present = int(any(contains_voice_hint(record) for record in selected_records))
        music_present = int(any(record["pool"].endswith("music") for record in selected_records))
        voice_fake = int(any(
            record["pool"] == "fake_voice" or
            (record["pool"] == "fake_music" and contains_voice_hint(record))
            for record in selected_records
        ))
        music_fake = int(any(record["pool"] == "fake_music" for record in selected_records))
        file_fake = max(voice_fake, music_fake)

        rows.append({
            "ID": identifier, "recipe_type": recipe_type, "duration": duration,
            "codec": codec, "bitrate": bitrate or "lossless", "channels": 2 if stereo else 1,
            "telephone": phone, "layout": layout,
            "sources": "|".join(str(record["source_id"]) for record in selected_records),
        })
        truth_rows.append({
            "ID": identifier, "FILE_FAKE": file_fake, "VOICE_FAKE": voice_fake,
            "MUSIC_FAKE": music_fake, "VOICE_PRESENT": voice_present, "MUSIC_PRESENT": music_present,
        })

    sample = pd.DataFrame({"ID": [row["ID"] for row in rows]})
    for column in PREDICTION_COLUMNS:
        sample[column] = 0.0
    sample.to_csv(OOD_ROOT / "data" / "sample_submission.csv", index=False)
    pd.DataFrame(rows).to_csv(OOD_ROOT / "ood_manifest.csv", index=False)
    pd.DataFrame(truth_rows).to_csv(OOD_ROOT / "ground_truth.csv", index=False)
    assert len(find_audio(test_dir)) == OOD_SIZE
    print("OOD dataset:", OOD_ROOT)
    display(pd.crosstab(pd.DataFrame(rows).recipe_type, pd.DataFrame(rows).codec, margins=True))
    display(pd.DataFrame(rows)[["duration", "channels", "telephone"]].describe(include="all"))


## 9. 같은 `script.py`로 OOD 2,500개 추론

제출용 코드와 OOD 평가용 코드가 달라지는 것을 막기 위해, 수정된 `STAGE/script.py`를 그대로 실행합니다. 2,500개는 DACON의 1,200개보다 많으므로 실행 시간이 50분을 넘을 수 있습니다.


In [ ]:
if RUN_OOD_EVALUATION:
    if not torch.cuda.is_available():
        raise RuntimeError("OOD 전체 ensemble 평가는 GPU가 필요합니다")
    started = time.perf_counter()
    subprocess.run([sys.executable, str(STAGE / "script.py")], cwd=OOD_ROOT, check=True)
    elapsed_minutes = (time.perf_counter() - started) / 60
    generated = OOD_ROOT / "output" / "submission.csv"
    if not generated.is_file():
        raise FileNotFoundError(generated)
    predictions = pd.read_csv(generated)
    if predictions["ID"].astype(str).tolist() != pd.read_csv(
        OOD_ROOT / "data" / "sample_submission.csv"
    )["ID"].astype(str).tolist():
        raise RuntimeError("OOD prediction ID/order mismatch")
    shutil.copy2(generated, OOD_PREDICTIONS)
    print(f"OOD 2,500 inference: {elapsed_minutes:.2f} minutes")
    print("saved:", OOD_PREDICTIONS)


## 10. DACON 공식 지표 계산

공식 페이지와 동일하게 FAKE=1로 두고, Voice EER은 voice가 존재하는 sample, Music EER은 music이 존재하는 sample에서만 계산합니다.


In [ ]:
def official_eer(y_true, y_score):
    y_true = np.asarray(y_true, dtype=int)
    y_score = np.asarray(y_score, dtype=float)
    if np.unique(y_true).size != 2:
        raise ValueError("EER requires both classes")
    fpr, tpr, _ = roc_curve(y_true, y_score, pos_label=1, drop_intermediate=False)
    fnr = 1 - tpr
    index = np.argmin(np.abs(fpr - fnr))
    return float((fpr[index] + fnr[index]) / 2)


def official_metrics(truth: pd.DataFrame, prediction: pd.DataFrame) -> dict:
    merged = truth.merge(prediction, on="ID", validate="one_to_one")
    voice = merged.VOICE_PRESENT.eq(1)
    music = merged.MUSIC_PRESENT.eq(1)
    report = {
        "file_eer": official_eer(merged.FILE_FAKE, merged.FILE_FAKE_PROB),
        "voice_eer": official_eer(merged.loc[voice, "VOICE_FAKE"], merged.loc[voice, "VOICE_FAKE_PROB"]),
        "music_eer": official_eer(merged.loc[music, "MUSIC_FAKE"], merged.loc[music, "MUSIC_FAKE_PROB"]),
        "voice_presence_auc": float(roc_auc_score(merged.VOICE_PRESENT, merged.VOICE_PRESENT_PROB)),
        "music_presence_auc": float(roc_auc_score(merged.MUSIC_PRESENT, merged.MUSIC_PRESENT_PROB)),
    }
    report["ads"] = (
        .5 * (1 - report["file_eer"]) +
        .2 * (1 - report["voice_eer"]) +
        .3 * (1 - report["music_eer"])
    )
    report["cps"] = .5 * report["voice_presence_auc"] + .5 * report["music_presence_auc"]
    report["score"] = .9 * report["ads"] + .1 * report["cps"]
    return report


if RUN_OOD_EVALUATION:
    truth = pd.read_csv(OOD_ROOT / "ground_truth.csv")
    prediction = pd.read_csv(OOD_PREDICTIONS)
    report = official_metrics(truth, prediction)
    OOD_REPORT.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
    display(pd.DataFrame([report]).T.rename(columns={0: "value"}))
    print("주의: music 내부 voice label은 caption keyword hint입니다.")
    print("saved:", OOD_REPORT)


## 11. L4 50분 시간 게이트

OOD 중 길이가 긴 파일을 우선하여 별도 process로 실행합니다. 모델 로드 시간까지 포함한 파일당 시간에 15%를 더해 1,200개를 환산합니다. L4가 아니거나 50분을 넘으면 제출 ZIP 생성을 중단하는 것이 기본값입니다.


In [ ]:
TIME_GATE_PATH = PROJECT / "l4_time_gate.json"
time_gate_passed = False

if RUN_L4_BENCHMARK:
    if not torch.cuda.is_available() or "L4" not in torch.cuda.get_device_name(0).upper():
        raise RuntimeError(f"L4에서 시간 측정해야 합니다: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
    manifest_frame = pd.read_csv(OOD_ROOT / "ood_manifest.csv")
    benchmark_rows = (
        manifest_frame.sort_values(["duration", "telephone"], ascending=[False, False])
        .groupby("codec", group_keys=False).head(max(1, BENCHMARK_FILES // manifest_frame.codec.nunique()))
        .sort_values("duration", ascending=False).head(BENCHMARK_FILES)
    )
    benchmark_root = LOCAL / "benchmark"
    if benchmark_root.exists():
        shutil.rmtree(benchmark_root)
    (benchmark_root / "data" / "test").mkdir(parents=True)
    source_lookup = {p.stem: p for p in find_audio(OOD_ROOT / "data" / "test")}
    for identifier in benchmark_rows.ID.astype(str):
        source = source_lookup[identifier]
        os.symlink(source, benchmark_root / "data" / "test" / source.name)
    benchmark_sample = pd.DataFrame({"ID": benchmark_rows.ID.astype(str).tolist()})
    for column in PREDICTION_COLUMNS:
        benchmark_sample[column] = 0.0
    benchmark_sample.to_csv(benchmark_root / "data" / "sample_submission.csv", index=False)

    started = time.perf_counter()
    subprocess.run([sys.executable, str(STAGE / "script.py")], cwd=benchmark_root, check=True)
    measured = (time.perf_counter() - started) / 60
    projected = measured / len(benchmark_rows) * TARGET_TEST_FILES * TIME_SAFETY_FACTOR
    time_gate = {
        "gpu": torch.cuda.get_device_name(0), "benchmark_files": len(benchmark_rows),
        "measured_minutes": measured, "safety_factor": TIME_SAFETY_FACTOR,
        "projected_1200_minutes": projected, "limit_minutes": TIME_LIMIT_MINUTES,
        "passed": projected <= TIME_LIMIT_MINUTES,
    }
    TIME_GATE_PATH.write_text(json.dumps(time_gate, indent=2), encoding="utf-8")
    print(json.dumps(time_gate, indent=2))
    time_gate_passed = bool(time_gate["passed"])
    if ENFORCE_50_MIN_GATE and not time_gate_passed:
        raise RuntimeError(
            "50분 안전 게이트 실패. MAX_SEPARATION_SECONDS를 18 또는 MAX_DF_SEGMENTS를 2로 줄여 "
            "section 5부터 다시 실행하고 성능/시간을 재검증하세요."
        )
else:
    print("시간 gate가 실행되지 않았습니다. 실제 제출 전 반드시 L4에서 실행하세요.")


## 12. `submit.zip` 생성과 구조·용량 검사

최상위에는 `model/`, `script.py`, `requirements.txt`만 포함합니다. 압축 10 GB, 해제 32 GB를 자동 검사합니다.


In [ ]:
if BUILD_SUBMIT_ZIP:
    if ENFORCE_50_MIN_GATE and not time_gate_passed:
        raise RuntimeError("L4 50분 gate를 통과해야 submit.zip을 생성합니다")

    OUTPUT_ZIP.parent.mkdir(parents=True, exist_ok=True)
    if OUTPUT_ZIP.exists():
        OUTPUT_ZIP.unlink()
    with zipfile.ZipFile(
        OUTPUT_ZIP, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=1, allowZip64=True
    ) as archive:
        for top_name in ("model", "script.py", "requirements.txt"):
            source = STAGE / top_name
            if source.is_dir():
                for path in sorted(source.rglob("*")):
                    if path.is_file() and "__pycache__" not in path.parts:
                        archive.write(path, path.relative_to(STAGE).as_posix())
            else:
                archive.write(source, top_name)

    with zipfile.ZipFile(OUTPUT_ZIP) as archive:
        bad = archive.testzip()
        if bad:
            raise RuntimeError(f"corrupt ZIP member: {bad}")
        names = archive.namelist()
        top_levels = {name.split("/", 1)[0] for name in names}
        uncompressed = sum(info.file_size for info in archive.infolist())
    compressed = OUTPUT_ZIP.stat().st_size
    if top_levels != {"model", "script.py", "requirements.txt"}:
        raise RuntimeError(f"wrong top-level: {top_levels}")
    for marker in ("model/df_arena_1b/", "model/htdemucs/", "model/panns/", "model/aasist_ensemble/"):
        if not any(name.startswith(marker) for name in names):
            raise RuntimeError(f"missing packaged asset: {marker}")
    if compressed >= 10 * 2**30:
        raise RuntimeError(f"ZIP 10GB 초과: {compressed / 2**30:.2f} GiB")
    if uncompressed >= 32 * 2**30:
        raise RuntimeError(f"해제 32GB 초과: {uncompressed / 2**30:.2f} GiB")

    print("created:", OUTPUT_ZIP)
    print("compressed GiB:", round(compressed / 2**30, 3))
    print("uncompressed GiB:", round(uncompressed / 2**30, 3))
    print("SHA256:", sha256(OUTPUT_ZIP))


## 13. 선택 사항: DACON 제공 dummy data smoke test

`open.zip`에 dummy `data/`가 포함되어 있으면 최종 ZIP을 다시 풀어 실제 실행 구조까지 확인합니다. 이 검사는 정확도나 1,200개 시간 제한을 대신하지 않습니다.


In [ ]:
RUN_DACON_DUMMY_SMOKE_TEST = False

if RUN_DACON_DUMMY_SMOKE_TEST:
    data_candidates = [p.parent for p in baseline_extract.rglob("sample_submission.csv")]
    if len(data_candidates) != 1:
        raise FileNotFoundError(f"open.zip dummy data를 하나 찾아야 합니다: {data_candidates}")
    smoke = LOCAL / "dacon_smoke"
    if smoke.exists():
        shutil.rmtree(smoke)
    safe_extract(OUTPUT_ZIP, smoke)
    shutil.copytree(data_candidates[0], smoke / "data")
    subprocess.run([sys.executable, "script.py"], cwd=smoke, check=True)
    result = pd.read_csv(smoke / "output" / "submission.csv")
    sample = pd.read_csv(smoke / "data" / "sample_submission.csv")
    assert list(result.columns) == ["ID", *PREDICTION_COLUMNS]
    assert result.ID.astype(str).tolist() == sample.ID.astype(str).tolist()
    values = result[PREDICTION_COLUMNS].to_numpy(float)
    assert np.isfinite(values).all() and ((values >= 0) & (values <= 1)).all()
    print("DACON dummy smoke test: PASS")


## 실행 순서와 결과물

1. L4 GPU를 선택하고 `BASELINE_OR_OPEN_ZIP`, `AASIST_BEST_PT` 경로를 확인합니다.
2. 위에서 아래로 실행합니다. 원본 archive 총량이 크므로 Drive 또는 Colab 여유 공간을 먼저 확인합니다.
3. OOD 공식 지표는 `ood2500_official_metrics.json`, 예측은 `ood2500_predictions.csv`에 저장됩니다.
4. 시간 gate를 통과하면 `deepvoice_ensemble_ood2500/submit.zip`이 생성됩니다.
5. 첫 제출 전 `RUN_DACON_DUMMY_SMOKE_TEST=True`로 구조 검사를 권장합니다.

OOD 음원 자체는 자동으로 Drive에 복사하지 않습니다. 약 24 GB의 archive는 `archives/`에 보존되고, 변환된 2,500개는 빠른 추론을 위해 `/content`에 생성됩니다. 필요하면 manifest와 ground truth만 Drive에 복사하세요.
